# Exploring Applicant Data

# Introduction

When someone decides to pursue data science at WQU, the very first step is
creating an account. After that, they must take an **admissions exam** to
begin their journey. But here is the problem: **not everyone who registers
actually completes the exam.** Every applicant who signs up and then drops
off before finishing is a missed opportunity — for them and for the program.

> ❓ **The question that drives this whole project:** *Can a simple nudge — an
> email reminder — increase the share of applicants who finish the admissions
> exam?*

To answer it responsibly we cannot just send the email and eyeball the
result. We need a designed **experiment** — an **A/B test** (also called a
**hypothesis test**) — in which one group receives the reminder and a
comparable group does not, so that any difference we observe can be
attributed to the email rather than to chance.

This is the **first of four notebooks**. Before we can design that
experiment, we must understand **who applies** to the DS Lab. This lesson is
pure exploratory data analysis (EDA): where applicants come from, how old
they are, and what educational backgrounds they bring. Everything we learn
here lays the groundwork for the experimental design (Notebook 2), the
statistical test (Notebook 3), and the dashboard (Notebook 4).

> 📌 **Data ethics.** This project is based on a real experiment run by the
> WQU data science team in **June 2022**. To protect privacy you will work
> with **synthetic data** — engineered to resemble the real dataset's
> statistical characteristics without exposing any real names, birthdays, or
> email addresses. The habit of asking *"should I even be looking at this
> field?"* is one you should carry into every real project.

By the end of this notebook you will be able to:

-   Connect to a MongoDB database and access collections using PyMongo.
-   Retrieve and inspect semi-structured documents from a MongoDB
    collection.
-   Aggregate data using MongoDB aggregation pipelines (`$group`,
    `$project`, `$dateDiff`).
-   Convert MongoDB query results into pandas DataFrames and Series.
-   Enrich data using the `country_converter` library.
-   Create bar charts, histograms, and choropleth maps using Plotly
    Express.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1183284457", h="3298dbabb7", width=700, height=450) 

# 1. Conceptual Foundation

## Project context: what we're trying to build

This project spans **four notebooks**, and it helps to keep the whole arc in
view from the start:

| Notebook | Role in the experiment | What you build |
|---|---|---|
| **1 — this one** | 🔍 *Understand the population* | EDA on applicant records (country, age, education) |
| **2 — ETL** | 🔄 *Prepare & assign* | Pipelines that clean data and split applicants into groups |
| **3 — Hypothesis test** | 🧪 *Decide* | A chi-square test that says whether the email worked |
| **4 — Dashboard** | 📦 *Share* | An interactive Dash app for stakeholders |

In **this** notebook you will answer three descriptive questions about the
DS Lab applicant records stored in MongoDB:

1.  **Where are applicants from?** — aggregate by country and visualize with
    bar charts and a choropleth map.
2.  **How old are applicants?** — compute ages from birthdates and plot the
    distribution.
3.  **What education levels do applicants have?** — count applicants by
    highest degree earned and visualize the result.

> 🧠 **Why describe before we experiment?** You cannot design a fair
> intervention for a population you do not understand. If, say, most
> applicants come from a handful of countries or cluster in a narrow age
> band, that shapes how we interpret the experiment later. EDA is the
> reconnaissance that makes the experiment trustworthy.

➡️ First, we need a way to *reach* the data. The applicant records do not
live in a CSV — they live in a **MongoDB** database, so we begin there.

## MongoDB and PyMongo basics

So far in the course your data has arrived as flat files (CSV) or relational
tables. The DS Lab applicant data is different: it lives in **MongoDB**, a
**NoSQL** database that stores records as flexible, JSON-like **documents**
grouped into **collections**.

> 🧠 **The mental model: documents, not rows.**
>
> | SQL world | MongoDB world |
> |---|---|
> | Database | Database |
> | Table | **Collection** |
> | Row | **Document** (a JSON-like dict) |
> | Column | **Field** |
> | Rigid schema (every row identical) | **Semi-structured** (documents may vary) |
>
> A document looks just like a Python dictionary: `{"firstName": "Alice",
> "countryISO2": "NG", "birthday": ...}`. "Semi-structured" means most
> documents share the same fields, but the database does not *force* them to —
> a flexibility that is powerful and occasionally a trap (a field you assume
> exists may be missing in some documents).

To talk to MongoDB from Python we use the **PyMongo** library. The workflow
is always the same three moves:

1.  **Connect** to the server by creating a `MongoClient`.
2.  **Select** a database, then a collection inside it.
3.  **Query** documents with methods like `find_one()`, `find()`, or
    `aggregate()`.

> 🔧 **Instruction:** locate the IP address of the machine running MongoDB
> and assign it to `MONGODB_HOST` as a **string** (wrap the IP in quotes).
>
> ⚠️ **The IP is dynamic** — it may change every time you start the lab.
> Always re-check the current IP before connecting, or your `MongoClient`
> will silently fail to reach the server.

<figure>
<img src="attachment:images/mongo_ip.png" alt="MongoDB" />
<figcaption aria-hidden="true">MongoDB</figcaption>
</figure>

Let's see the smallest possible connection example:

In [ ]:
# Minimal PyMongo connection example (toy demonstration)
import pandas as pd
from wqulibs.database import reset
from pymongo import MongoClient

MONGODB_HOST = "localhost"

# Connect to the local MongoDB server
demo_client = MongoClient(host=MONGODB_HOST, port=27017)
print("Type of client:", type(demo_client))

# List available databases
print("Databases:", demo_client.list_database_names())

🔍 **What just happened:** `MongoClient(host=..., port=27017)` opened a
connection to the server, and `list_database_names()` asked it what databases
it holds. The printed type confirms `demo_client` is a live `MongoClient`
object — our handle to everything else. `27017` is MongoDB's default port.

Once connected, you drill down to a collection with **bracket notation**,
exactly like indexing nested dictionaries:

In [ ]:
# Access a database and collection
demo_db = demo_client["wqu-abtest"]
demo_collection = demo_db["ds-applicants"]
print("Collection type:", type(demo_collection))

Before we can query anything, the demo collection has to actually contain
data. The `reset()` helper (drops and) repopulates it from a JSON file so we
start from a known, clean state:

In [ ]:
reset(demo_collection)

### Retrieving documents

With a populated collection in hand, the two most basic queries are:

-   `count_documents(filter)` — how many documents match the filter.
-   `find_one(filter)` — return a single matching document.

The **empty filter `{}` matches everything**, so `count_documents({})` is the
total size of the collection and `find_one({})` grabs an arbitrary document so
you can inspect its shape.

In [ ]:
# Count all documents in the collection
n = demo_collection.count_documents({})
print("Total documents:", n)

# Retrieve one document
one_doc = demo_collection.find_one({})
print("Document type:", type(one_doc))
print("Document keys:", list(one_doc.keys()))

📊 **Reading the output:** the document count tells you the collection's size,
and `list(one_doc.keys())` reveals the **fields every applicant record
carries** — things like `birthday`, `countryISO2`, and `highestDegreeEarned`.
Peeking at one document's keys before writing analysis code is the MongoDB
equivalent of `df.head()`: it tells you what you have to work with.

➡️ Counting and fetching one document is fine for a peek, but to *summarize*
thousands of documents we need something more powerful: aggregation
pipelines.

### Aggregation pipelines

`count_documents` and `find_one` answer simple questions. To compute
group-level summaries — *how many applicants per country?*, *how old is each
applicant?* — we use MongoDB's **aggregation pipeline**.

> 🧠 **The pipeline metaphor.** Think of `aggregate()` as an assembly line.
> Documents flow in one end and pass through an ordered list of **stages**,
> each of which transforms the stream before handing it to the next. You
> describe the line as a Python list of stage-dictionaries.

Two stages do most of the work in this notebook:

| Stage | What it does | Example |
|---|---|---|
| `$group` | Buckets documents by a field and computes a summary per bucket | count applicants per country |
| `$project` + `$dateDiff` | Reshapes each document into derived fields | compute age from `birthday` |

> ⚠️ **Note on the `$` prefix.** Inside a pipeline, strings like `$group`,
> `$project`, `$count`, and `$dateDiff` are **MongoDB operators**, and a field
> reference like `$countryISO2` means "the value of the `countryISO2` field."
> The leading `$` is MongoDB syntax — it is *not* a typo and *not* math.

> ❗️ **But wait — cursors are single-use.** The object returned by
> `aggregate()` (and `find()`) is a **cursor**, not a list. A cursor is a
> one-time stream: the moment you iterate it — via `list()`, `pd.DataFrame()`,
> or a `for` loop — it is **exhausted**. Reuse the same variable and it will
> look empty. To read the data again, **re-run the `aggregate()` call**. This
> single fact is the cause of most "why is my DataFrame empty?" confusion in
> this lesson.

Here is a `$group` pipeline that counts documents by a field:

In [ ]:
# Aggregation: count applicants by country (first 3 results)
pipeline = [
    {"$group": {"_id": "$countryISO2", "count": {"$count": {}}}},
    {"$limit": 3}
]
sample_agg = list(demo_collection.aggregate(pipeline))
print("Sample aggregation result:")
for doc in sample_agg:
    print(doc)

📊 **Reading the output:** each result document has the form
`{"_id": <country code>, "count": <n>}`. `$group` put the value of
`$countryISO2` into `_id` (the grouping key) and `$count: {}` tallied how many
documents fell into each bucket. We wrapped the cursor in `list()` so we could
print it — which also **exhausts** it.

And here is a `$project` + `$dateDiff` pipeline that derives a new field
(age) from an existing one (birthday):

In [ ]:
# Aggregation: compute age in years for first 3 applicants
pipeline_age = [
    {
        "$project": {
            "years": {
                "$dateDiff": {
                    "startDate": "$birthday",
                    "endDate": "$$NOW",
                    "unit": "year",
                }
            }
        }
    },
    {"$limit": 3}
]
sample_ages = list(demo_collection.aggregate(pipeline_age))
print("Sample age computation:")
for doc in sample_ages:
    print(doc)

📊 **Reading the output:** `$dateDiff` measured the distance from each
applicant's `$birthday` to `$$NOW` (the server's current time) in years,
exposing it as a new `years` field. `$project` controls which fields survive
into the output — here, only `years`. This is exactly how we will compute the
age distribution later.

➡️ Aggregation results come back as plain Python lists of dicts. To analyze
and plot them, we lift them into pandas.

### Turning results into pandas objects

An aggregation result is just a list of dictionaries, and `pd.DataFrame()`
turns a list of dicts into a tidy table in one step — each dict becomes a row,
each key becomes a column:

In [ ]:
import pandas as pd

# Convert aggregation result to DataFrame
df_demo = pd.DataFrame(sample_ages)
print(df_demo)

✅ **Sanity check:** the DataFrame's columns (`_id`, `count`) match the keys
of the result dictionaries. From here, every pandas tool you already know —
`rename`, `sort_values`, plotting — is available.

➡️ With the data in pandas, we can visualize it. This notebook uses Plotly
Express for interactive charts.

## Visualizing with Plotly Express

You will build three kinds of figures with **Plotly Express**, each matched
to a different question:

| Chart | Function | Best for |
|---|---|---|
| Bar chart | `px.bar` | comparing counts across categories (countries, degrees) |
| Histogram | `px.histogram` | the distribution of a continuous variable (age) |
| Choropleth | `px.choropleth` | geographic data shaded on a world map |

> 💡 **Why Plotly here?** Unlike static matplotlib figures, Plotly charts are
> **interactive** — you can hover for exact values and zoom. That matters
> because in Notebook 4 these same figures will be embedded in a live
> dashboard.

Here is a minimal horizontal bar chart:

In [ ]:
import plotly.express as px

# Toy bar chart
toy_data = pd.DataFrame({
    "fruit": ["Apple", "Banana", "Cherry"],
    "count": [10, 7, 3]
})
fig = px.bar(
    toy_data,
    x="count",
    y="fruit",
    orientation="h",
    title="Toy Bar Chart"
)
fig.update_layout(xaxis_title="Count", yaxis_title="Fruit")
fig.show()

🔍 **What to notice:** `orientation="h"` makes the bars horizontal, which
keeps long category labels (like full country names) readable. We map the
*count* to the x-axis and the *category* to the y-axis — the standard recipe
for the country and education charts ahead.

## Enriching data with `country_converter`

The applicant records store nationality as **two-letter ISO codes** (e.g.,
`"NG"` for Nigeria). Codes are compact but unreadable in a chart, and Plotly's
choropleth needs **three-letter** ISO codes. The `country_converter` library
bridges both gaps: code → full name, and ISO2 → ISO3.

> 📌 **A habit worth keeping — vet a library before you depend on it.** Before
> pulling in any open-source package, check:
>
> 1.  **License** — compatible with your project?
> 2.  **Maintenance** — actively maintained, or abandoned?
> 3.  **Quality** — tests, documentation, a real user base?
>
> `country_converter` passes all three, which is why we trust it here.

In [ ]:
from country_converter import CountryConverter

cc = CountryConverter()

# Convert ISO2 codes to short names
print(cc.convert("NG", to="name_short"))  # Nigeria
print(cc.convert("IN", to="name_short"))  # India

# Convert ISO2 to ISO3 (needed for Plotly choropleth)
print(cc.convert("NG", to="ISO3"))  # NGA

📊 **Reading the output:** one call converted `"NG"` to both the human name
`"Nigeria"` and the ISO3 code `"NGA"`. We will use `name_short` to label bars
and `ISO3` to place countries on the choropleth map.

## Using `PrettyPrinter`

MongoDB documents are nested. A flat `print()` smears them onto one line;
Python's built-in `pprint` indents them so the structure is legible:

In [ ]:
from pprint import PrettyPrinter

pp = PrettyPrinter(indent=2)
pp.pprint({"name": "Alice", "scores": [90, 85, 92]})

🔍 **What to notice:** `indent=2` gives each nesting level two spaces, so
nested lists and dicts line up vertically. When you inspect a real applicant
document in a moment, `pp.pprint(...)` will make its fields far easier to scan
than `print(...)`.

## Key points

-   [`MongoClient`](https://pymongo.readthedocs.io/en/stable/api/pymongo/mongo_client.html#pymongo.mongo_client.MongoClient)
    — connects to a MongoDB server.
-   [`Collection.find_one()`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.find_one)
    — retrieves a single document.
-   [`Collection.count_documents()`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.count_documents)
    — counts documents matching a filter.
-   [`Collection.aggregate()`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.aggregate)
    — runs an aggregation pipeline.
-   [`pd.DataFrame()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
    — creates a DataFrame from a list of dictionaries.
-   [`px.bar()`](https://plotly.com/python-api-reference/generated/plotly.express.bar.html)
    — creates bar charts.
-   [`px.histogram()`](https://plotly.com/python-api-reference/generated/plotly.express.histogram.html)
    — creates histograms.
-   [`px.choropleth()`](https://plotly.com/python-api-reference/generated/plotly.express.choropleth.html)
    — creates choropleth maps.
-   [`CountryConverter`](https://github.com/konstantinstadler/country_converter)
    — converts between country name formats.
-   [`PrettyPrinter`](https://docs.python.org/3/library/pprint.html#pprint.PrettyPrinter)
    — formats nested data structures for readable output.

➡️ Concepts in hand, we now switch from *reading about* the tools to *using*
them on the real applicant collection.

# Applied Exercises

## 2. Setup

It is good practice to gather every import in a single cell at the top of the
notebook, so a reader can see all dependencies at a glance and re-running from
scratch is painless.

**Code 7.1.2.1**:

In [ ]:
from pprint import PrettyPrinter

import pandas as pd
import plotly.express as px
from country_converter import CountryConverter
from pymongo import MongoClient

## 3. Connect to MongoDB

### Problem

Before we can analyze applicant data, we need a live connection to the MongoDB
server and a handle on the right database and collection. Recall the
hierarchy: a **server** hosts **databases**, each of which holds
**collections** of documents.

### Approach

Create a `MongoClient` pointed at the local server (`localhost`, port
`27017`), list the available databases to confirm the connection, select the
`"wqu-abtest"` database, and grab the `"ds-applicants"` collection. You will
also build a `PrettyPrinter` for readable document output.

Key variables you will define:

-   `pp` — a `PrettyPrinter` instance with `indent=2`.
-   `client` — a `MongoClient` connected to `localhost:27017`.
-   `db` — the `"wqu-abtest"` database.
-   `ds_app` — the `"ds-applicants"` collection (your main entry point for
    every query in this notebook).

### Tasks

Create a `PrettyPrinter` and assign it to `pp`.

**Code 7.1.3.1**:

In [ ]:
pp = PrettyPrinter(indent = 2)
print("pp type:", type(pp))

Create a `MongoClient` that connects to `localhost` on port `27017`.

**Code 7.1.3.2**:

In [ ]:
client = MongoClient(host=MONGODB_HOST, port=27017)
print("client type:", type(client))

Print the list of databases available on the server. You should see
`"wqu-abtest"` among the results.

**Code 7.1.3.3**:

In [ ]:
# list available databases using pp
pp.pprint(client.list_database_names())

Access the `"wqu-abtest"` database and the `"ds-applicants"` collection. The
variable `ds_app` will be your main entry point for all queries in this
notebook.

**Code 7.1.3.4**:

In [ ]:
db = client["wqu-abtest"]
ds_app = db["ds-applicants"]
print("ds_app type:", type(ds_app))

### Checkpoint

🔍 **What this guards:** the asserts below confirm all four handles are the
*right type* before you build on them — `pp` is a `PrettyPrinter`, `client` is
a `MongoClient`, and `ds_app` actually points at the `ds-applicants`
collection. Catching a wrong connection here saves you from cryptic errors
three sections later.

In [ ]:
assert str(type(pp)) == "<class 'pprint.PrettyPrinter'>", (
    f"Expected pp to be a PrettyPrinter, got {type(pp)}"
)
assert "MongoClient" in str(type(client)), (
    f"Expected client to be a MongoClient, got {type(client)}"
)
assert ds_app.name == "ds-applicants", (
    f"Expected collection name 'ds-applicants', got '{ds_app.name}'"
)
print("All checks passed!")

✅ **Passed.** The connection is live and every handle is what we expect. Every
query for the rest of the notebook flows through `ds_app`.

➡️ Now that we can reach the collection, let's see how big it is and what a
single record looks like.

## 4. Explore the Collection

### Problem

We need a feel for the dataset's **size** and **shape** before analyzing it.
Because MongoDB documents are semi-structured, inspecting one sample document
reveals the fields available (birthday, country, degree, …) and their types.

### Approach

Use `count_documents({})` for the total record count, then `find_one({})` to
pull one document and pretty-print it. Store the count in `n_documents` and the
sample in `result`.

### Tasks

Count the total number of documents in the `ds_app` collection.

**Code 7.1.4.1**:

In [ ]:
n_documents = ds_app.count_documents({})  # count all documents with an empty filter
print("Documents in 'ds-applicants':", n_documents)

Retrieve a single document from the collection and pretty-print it to see the
available fields. You should see fields like `birthday`, `countryISO2`,
`email`, `firstName`, `highestDegreeEarned`, and `lastName`.

**Code 7.1.4.2**:

In [ ]:
result = ds_app.find_one({})  # retrieve a single document
pp.pprint(result)

📊 **Reading the document:** the pretty-printed record exposes the full field
set we have to work with. Three fields drive the rest of this notebook:
`countryISO2` (§5), `birthday` (§6, via `$dateDiff`), and `highestDegreeEarned`
(§7). Notice the personal fields (`email`, `firstName`, `lastName`) too —

> 📌 **Tip / ethics reminder.** Real applicant records carry names, emails,
> and birthdays — exactly the kind of personal data that demands careful
> handling. Here the data is synthetic so there is no privacy risk, but the
> instinct to minimize and protect personal fields should travel with you to
> every real dataset.

### Checkpoint

🔍 **What this guards:** that `n_documents` is a positive integer, that
`result` is a dict, and that the document carries the three keys we depend on.
If the `expected_keys` check fails, the schema is not what the analysis
assumes — a problem worth catching now.

In [ ]:
assert isinstance(n_documents, int) and n_documents > 0, (
    f"Expected n_documents to be a positive integer, got {n_documents}"
)
assert isinstance(result, dict), (
    f"Expected result to be a dict, got {type(result)}"
)
expected_keys = {"countryISO2", "birthday", "highestDegreeEarned"}
actual_keys = set(result.keys())
assert expected_keys.issubset(actual_keys), (
    f"Expected document to contain keys {expected_keys}, "
    f"but got {actual_keys}"
)
print(f"Collection has {n_documents} documents. All checks passed!")

✅ **Passed.** We know the collection's size and have confirmed the fields our
analysis relies on actually exist.

➡️ With the schema confirmed, we answer the first descriptive question: where
do applicants come from?

## 5. Applicant Nationality

### Problem

The geographic spread of applicants matters for designing interventions (like
email reminders) that might need to account for regional differences — time
zones, languages, connectivity. We will count applicants per country, convert
codes to readable names, compute proportions, and produce both bar charts and
a choropleth map.

### Approach

You will:

1.  Run a `$group` aggregation to count applicants by `countryISO2`.
2.  Convert the result to a DataFrame `df_nationality` with columns
    `"country_iso2"` and `"count"`, sorted by `"count"` ascending.
3.  Use `CountryConverter` to add `"country_name"` and `"country_iso3"`
    columns.
4.  Add a `"count_pct"` column for proportions.
5.  Create horizontal bar charts (raw counts and percentages) and a
    choropleth map.

> ❗️ **But wait — why proportions as well as counts?** A raw count of, say,
> 400 applicants is hard to judge in isolation. A **proportion** puts it on a
> common 0–1 scale that sums to 1 across all countries:
>
> $$\text{count\_pct}_i = \frac{\text{count}_i}{\sum_j \text{count}_j}$$
>
> where `count`$_i$ is the number of applicants from country $i$. Proportions
> let you say "this country is 12% of all applicants" — a statement that
> survives the dataset growing or shrinking.

### Tasks

Use the `aggregate()` method with `$group` to count applicants by
`countryISO2`. Store the cursor in `result`. The pipeline should group on
`"$countryISO2"` and use `{"$count": {}}` to compute the count.

**Code 7.1.5.1**:

In [ ]:
result = ds_app.aggregate(
    [{"$group": {"_id": "$countryISO2", "count": {"$count": {}}}}]
)
print("result type:", type(result))

Convert the aggregation result into a DataFrame named `df_nationality`. Rename
the `"_id"` column to `"country_iso2"` and sort by `"count"` in ascending
order. You should see two columns: `"country_iso2"` and `"count"`.

**Code Task 7.1.5.2**:

In [ ]:
df_nationality = (
    pd.DataFrame(result)
    .rename(columns={'_id': 'country_iso2'})
    .sort_values("count")
)
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()

> ⚠️ **If `df_nationality` looks empty**, the aggregation cursor from the
> previous cell was already consumed — MongoDB cursors iterate only once
> (recall the cursor caveat from §1). Re-run the `aggregate()` cell, then this
> cell again.

Instantiate a `CountryConverter` named `cc`, then add a `"country_name"`
column to `df_nationality` by converting the `"country_iso2"` column to short
names.

**Code 7.1.5.3**:

In [ ]:
cc = CountryConverter()
df_nationality["country_name"] = cc.convert(
    df_nationality["country_iso2"], # pass the column with ISO2 codes
    to="name_short" # convert to short names
)
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()

With readable country names attached, plot the **10 countries with the most
applicants**. Because `df_nationality` is sorted ascending, the top 10 are the
**last** 10 rows — `df_nationality.tail(10)`. Label the x-axis
`"Frequency [count]"`, the y-axis `"Country"`, and title it
`"DS Applicants by Country"`.

**Code 7.1.5.4**:

In [ ]:
fig = px.bar(
    data_frame=df_nationality.tail(10), # last 10 rows
    x="count",
    y="country_name",
    orientation="h",
    title="DS Applicants by Country",
)
fig.update_layout(
    xaxis_title="Frequency [count]",
    yaxis_title="Country"
)
fig.show()

📊 **Reading the chart:** the bars are sorted, so the country with the most
applicants sits at the top. Look at how *concentrated* the distribution is — if
a few countries dominate, the applicant pool is geographically lopsided, which
is worth remembering when we later assume groups are comparable. (Specific
counts here are illustrative; read the shape, not exact numbers.)

Now express the same information as proportions. Add a `"count_pct"` column —
each country's count divided by the total — using the formula from the
**Approach** above.

**Code Task 7.1.5.5**:

In [ ]:
# compute proportion: count divided by total
df_nationality['count_pct'] = (
    df_nationality['count'] / df_nationality['count'].sum()
)
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()

Recreate the top-10 bar chart, this time using `"count_pct"`. Label the x-axis
`"Frequency [%]"`, the y-axis `"Country"`, and keep the title
`"DS Applicants by Country"`.

**Code 7.1.5.6**:

In [ ]:
fig = px.bar(
    data_frame=df_nationality.tail(10),
    x="count_pct", # use the percentage column
    y="country_name",
    orientation="h",
    title="DS Applicants by Country",
)
fig.update_layout(
    xaxis_title="Frequency [%]",
    yaxis_title="Country"
)
fig.show()

📊 **Reading the chart:** the *shape* is identical to the count chart — only
the x-axis units changed from absolute counts to shares of the total. The
payoff is interpretability: you can now read each bar as "this percent of all
applicants," independent of the dataset's size.

To place these countries on a world map, Plotly needs **ISO3** codes. Add a
`"country_iso3"` column by converting `"country_iso2"`.

**Code Task 7.1.5.7**:

In [ ]:
df_nationality['country_iso3'] = cc.convert(
    df_nationality['country_iso2'],  # pass the column with ISO2 codes
    to='ISO3'  # convert to three-letter ISO codes
)
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()

Create a choropleth map of applicant counts by country. Use `"natural earth"`
as the projection, `px.colors.sequential.Oranges` as the color scale, and the
title `"DS Applicants: Nationalities"`.

**Code 7.1.5.8**:

In [ ]:
fig = px.choropleth(
    data_frame=df_nationality,
    locations="country_iso3", # column with ISO3 codes
    color="count",
    projection="natural earth",
    color_continuous_scale=px.colors.sequential.Oranges, # check colors from px.colors.sequential
    title="DS Applicants: Nationalities",
)
fig.show()

📊 **Reading the map:** a choropleth shades each country by its applicant
count — darker oranges mean more applicants. The map answers a question a bar
chart cannot: *is interest clustered in one region of the world, or spread
across continents?* That geographic pattern is context for every later
decision about the experiment.

> 📌 **Note on borders.** Political boundaries are subject to change, debate,
> and dispute. The boundaries shown in
> [Plotly](https://plotly.com/python/map-configuration/) come from the
> [Natural Earth dataset](https://www.naturalearthdata.com/); see their
> [disputed-boundaries policy](https://www.naturalearthdata.com/about/disputed-boundaries-policy/).

### Checkpoint

🔍 **What this guards:** that `df_nationality` carries all five expected
columns (`country_iso2`, `count`, `country_name`, `count_pct`,
`country_iso3`) **and** that `count_pct` sums to ~1.0 — the proportion sanity
check from the Approach, now enforced in code.

In [ ]:
assert "country_iso2" in df_nationality.columns, (
    "df_nationality is missing the 'country_iso2' column"
)
assert "count" in df_nationality.columns, (
    "df_nationality is missing the 'count' column"
)
assert "country_name" in df_nationality.columns, (
    "df_nationality is missing the 'country_name' column"
)
assert "count_pct" in df_nationality.columns, (
    "df_nationality is missing the 'count_pct' column"
)
assert "country_iso3" in df_nationality.columns, (
    "df_nationality is missing the 'country_iso3' column"
)
pct_sum = df_nationality["count_pct"].sum()
assert abs(pct_sum - 1.0) < 0.01, (
    f"Expected count_pct to sum to ~1.0, got {pct_sum:.4f}"
)
print(
    f"df_nationality has {df_nationality.shape[0]} countries "
    f"and {df_nationality.shape[1]} columns. All checks passed!"
)

✅ **Passed.** The nationality table is complete and internally consistent
(proportions sum to 1). First question answered.

➡️ Next descriptive question: how old are applicants? For that we turn
birthdates into ages.

## 6. Applicant Age

### Problem

The age distribution shapes how we communicate with applicants and design
programs. Documents store **birthdates**, not ages, so we compute ages with a
MongoDB aggregation and then visualize their distribution.

### Approach

You will:

1.  Run a `$project` aggregation with `$dateDiff` to compute each applicant's
    age in years (from `"$birthday"` to `"$$NOW"`).
2.  Convert the result to a pandas Series named `ages`.
3.  Create a histogram with Plotly Express.

> 💡 **Why a histogram and not a bar chart here?** Age is a **continuous**
> quantity, so we care about its *shape* — where it peaks, how spread out it
> is, whether it is skewed. A histogram bins the values and shows that shape;
> a bar chart (for discrete categories) would not.

### Tasks

Use `aggregate()` with a `$project` stage to compute the age of each
applicant. The `$dateDiff` operator takes `startDate`, `endDate`, and `unit`.
Store the cursor in `result`.

**Code 7.1.6.1**:

In [ ]:
result = ds_app.aggregate(
    [
        {
            "$project": {
                "years": {
                    "$dateDiff": {
                        "startDate": "$birthday",
                        "endDate": "$$NOW",
                        "unit": "year",
                    }
                }
            }
        }
    ]
)
print("result type:", type(result))

Convert the aggregation result into a DataFrame and extract the `"years"`
column as a Series named `ages`. You should see integer age values.

**Code Task 7.1.6.2**:

In [ ]:
# convert result to DataFrame and extract the "years" column
ages = pd.DataFrame(result)['years']
print("ages type:", type(ages))
print("ages shape:", ages.shape)
ages.head()

Create a histogram of `ages` with 20 bins. Label the x-axis `"Age"`, the
y-axis `"Frequency [count]"`, and title it
`"Distribution of DS Applicant Ages"`.

**Code 7.1.6.3**:

In [ ]:
fig = px.histogram(
    x=ages,
    nbins=20,
    title="Distribution of DS Applicant Ages",
)
fig.update_layout(
    xaxis_title="Age",
    yaxis_title="Frequency [count]"
)
fig.show()

📊 **Reading the histogram:** look for **where the distribution peaks** (the
most common age band), **how wide** it is (a tight cluster vs. a broad range),
and whether it is **skewed** (a long tail toward older applicants is typical
for adult learners). The choice of 20 bins trades detail against smoothness —
too few hides structure, too many turns the histogram into noise.

### Checkpoint

🔍 **What this guards:** that `ages` is a non-empty Series and that every value
is physically plausible (`min >= 0`, `max < 150`). The bounds are a cheap guard
against a botched date calculation producing negative or absurd ages.

In [ ]:
assert isinstance(ages, pd.Series), (
    f"Expected ages to be a pd.Series, got {type(ages)}"
)
assert len(ages) > 0, "ages Series is empty"
assert ages.min() >= 0, (
    f"Expected all ages >= 0, but min age is {ages.min()}"
)
assert ages.max() < 150, (
    f"Expected reasonable max age, but got {ages.max()}"
)
print(
    f"ages has {len(ages)} values, "
    f"range [{ages.min()}, {ages.max()}]. All checks passed!"
)

✅ **Passed.** Ages are computed and sane. Second question answered.

➡️ Final descriptive question: what educational backgrounds do applicants
bring?

## 7. Educational Attainment

### Problem

Educational background contextualizes the DS Lab audience. The
`highestDegreeEarned` field records each applicant's highest degree. We will
count applicants per degree, sort the results in a **meaningful hierarchical
order** (not alphabetical), and visualize them.

### Approach

You will:

1.  Run a `$group` aggregation to count applicants by `highestDegreeEarned`.
2.  Convert the result to a Series `education` (degree as index, count as
    values).
3.  Define an `ed_sort()` function mapping degree names to a hierarchical
    order so the Series sorts from lowest to highest degree.
4.  Create a horizontal bar chart of the sorted result.

> ❗️ **But wait — why a custom sort?** Sorted alphabetically, `"Bachelor's
> degree"` would come before `"High School"`, which is nonsense for an ordered
> category. Degrees have a natural **rank** (high school → some college →
> bachelor's → master's → doctorate). `ed_sort()` encodes that rank so the
> chart reads in the order a human expects.

### Tasks

Use `aggregate()` with `$group` to count applicants by
`"$highestDegreeEarned"`. Store the cursor in `result`.

**Code 7.1.7.1**:

In [ ]:
result = ds_app.aggregate(
    [{"$group": {"_id": "$highestDegreeEarned", "count": {"$count": {}}}}]
)
print("result type:", type(result))

Convert the result into a Series named `education`. Rename the `"_id"` column
to `"highest_degree_earned"`, set it as the index, and squeeze the DataFrame
into a Series.

**Code 7.1.7.2**:

In [ ]:
education = (
    pd.DataFrame(result)
    .rename({"_id": "highest_degree_earned"}, axis=1)
    .set_index("highest_degree_earned")
    .squeeze()
)
print("education type:", type(education))
print("education shape:", education.shape)
education

The degree categories must be sorted hierarchically from lowest to highest
level — not alphabetically. Complete `ed_sort()`: it receives a pandas Index
(the degree names) and must return a list of integers giving the sort order.
Use a dictionary comprehension to map each degree to its position in the
`degrees` list.

**Code Task 7.1.7.3**:

In [ ]:
def ed_sort(counts):
    """Sort array `counts` from lowest to highest degree."""
    degrees = [
        "High School or Baccalaureate",
        "Some College (1-3 years)",
        "Bachelor's degree",
        "Master's degree",
        "Doctorate (e.g. PhD)",
    ]
    # create a dict mapping degree name -> position
    mapping = {degree: position for position, degree in enumerate(degrees)}
    # build a list of positions for each entry in counts
    sort_order = [mapping[degree] for degree in counts]
    return sort_order


education.sort_index(key=ed_sort, inplace=True)
education

🔍 **How the sort works:** `mapping` turns the ordered `degrees` list into
`{degree_name: rank}`. Passing `ed_sort` as the `key` to `sort_index` tells
pandas to order the index by each degree's *rank* rather than its text, so the
Series now runs low → high.

Create a horizontal bar chart of `education`. Label the x-axis
`"Frequency [count]"`, the y-axis `"Highest Degree Earned"`, and title it
`"DS Applicant Education Levels"`.

**Code 7.1.7.4**:

In [ ]:
fig = px.bar(
    x=education,
    y=education.index,
    orientation="h",
    title="DS Applicant Education Levels",
)
fig.update_layout(
    xaxis_title="Frequency [count]",
    yaxis_title="Highest Degree Earned"
)
fig.show()

📊 **Reading the chart:** because the y-axis is now in degree order, you can
read the educational profile of the applicant pool top to bottom. Look for
where the mass sits — a pool concentrated at bachelor's/master's level implies
a very different audience than one dominated by high-school leavers, and that
informs how the program (and our reminder email) should speak to applicants.

### Checkpoint

🔍 **What this guards:** that `education` is a 5-level Series whose index is in
the exact expected low-to-high order, and that the counts are positive. This
verifies the custom sort actually took effect.

In [ ]:
assert isinstance(education, pd.Series), (
    f"Expected education to be a pd.Series, got {type(education)}"
)
assert len(education) == 5, (
    f"Expected 5 education levels, got {len(education)}"
)
expected_order = [
    "High School or Baccalaureate",
    "Some College (1-3 years)",
    "Bachelor's degree",
    "Master's degree",
    "Doctorate (e.g. PhD)",
]
actual_order = list(education.index)
assert actual_order == expected_order, (
    f"Expected education index order:\n{expected_order}\n"
    f"Got:\n{actual_order}"
)
assert education.sum() > 0, "education counts should be positive"
print(
    f"education has {len(education)} levels, "
    f"total count: {education.sum()}. All checks passed!"
)

✅ **Passed.** Education levels are counted and correctly ordered. Third and
final descriptive question answered.

# Wrap-up

In this notebook you:

-   Connected to a MongoDB server and accessed the `"ds-applicants"`
    collection using PyMongo.
-   Inspected semi-structured documents to understand available fields.
-   Used MongoDB aggregation pipelines (`$group`, `$project`,
    `$dateDiff`) to summarize applicant data by country, age, and
    education level.
-   Converted aggregation results into pandas DataFrames and Series.
-   Enriched country codes with full names and ISO3 codes using
    `country_converter`.
-   Created horizontal bar charts, histograms, and a choropleth map
    using Plotly Express.
-   Built a custom sorting function for hierarchical categorical data.

> 🧠 **Where this leaves us.** EDA only *describes* who applies — it cannot
> tell us whether a reminder email *changes behavior*. To answer that we need
> to (1) prepare the data into a clean, repeatable pipeline and assign
> applicants to groups, and then (2) test the result statistically.

➡️ Next, you will build reusable **ETL pipelines** to extract, transform, and
load applicant data from MongoDB — and you will assign applicants to the
control and treatment groups that the experiment depends on.